# Clean DEM NoData values

This notebook replaces non-finite pixels and extreme DEM sentinels with a declared NoData value and updates the GeoTIFF metadata. Set `INPUT_DIR` to the directory containing the rasters. It defaults to `DRY_RUN=True`, so set it to `False` only after reviewing the printed changes. Reload the layer or refresh raster statistics in QGIS after cleaning.

In [5]:
from pathlib import Path
import os
import tempfile

import numpy as np
import rasterio

INPUT_DIR = Path("/Users/wm722/Documents/SCDs/data/Texas/Floyada_Texas")
NODATA = -999999.0
INVALID_THRESHOLD = -1e2
INVALID_SENTINELS = {-9999.0}
DRY_RUN = False


In [6]:
def clean_raster(path, nodata=None, invalid_threshold=None, dry_run=None):
    nodata = NODATA if nodata is None else float(nodata)
    invalid_threshold = INVALID_THRESHOLD if invalid_threshold is None else float(invalid_threshold)
    dry_run = DRY_RUN if dry_run is None else bool(dry_run)
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        original_nodata = src.nodata

    invalid = (
        ~np.isfinite(data)
        | (data < invalid_threshold)
        | np.isin(data, tuple(INVALID_SENTINELS))
    )
    before_valid = data[~invalid]
    if before_valid.size == 0:
        raise ValueError(f"No valid pixels found in {path}")

    cleaned = np.array(data, dtype=np.float32, copy=True)
    cleaned[invalid] = nodata
    profile.update(dtype="float32", count=1, nodata=float(nodata))

    if not dry_run:
        temporary_path = path.with_name(f".{path.name}.cleaning.tmp")
        try:
            with rasterio.open(temporary_path, "w", **profile) as dst:
                dst.write(cleaned, 1)
                dst.nodata = float(nodata)
            os.replace(temporary_path, path)
        finally:
            if temporary_path.exists():
                temporary_path.unlink()

    valid_after = cleaned[cleaned != nodata]
    print(
        f"{path.name}: invalid={int(invalid.sum())}, "
        f"original_nodata={original_nodata}, "
        f"valid min {before_valid.min():.6g} -> {valid_after.min():.6g}, "
        f"valid max {before_valid.max():.6g} -> {valid_after.max():.6g}, "
        f"written={not dry_run}"
    )


In [7]:
paths = sorted(INPUT_DIR.glob("*.tif"))
if not paths:
    raise FileNotFoundError(f"No .tif files found in {INPUT_DIR}")

for path in paths:
    clean_raster(path)


USGS_one_meter_x26y375_TX_Panhandle_B8_2017.tif: invalid=61941281, original_nodata=-999999.0, valid min 966.95 -> 966.95, valid max 991.86 -> 991.86, written=True
USGS_one_meter_x26y376_TX_Panhandle_B8_2017.tif: invalid=0, original_nodata=-999999.0, valid min 968.2 -> 968.2, valid max 998.66 -> 998.66, written=True
USGS_one_meter_x26y377_TX_Panhandle_B8_2017.tif: invalid=0, original_nodata=-999999.0, valid min 969.63 -> 969.63, valid max 1003.88 -> 1003.88, written=True
USGS_one_meter_x26y378_TX_Panhandle_B8_2017.tif: invalid=0, original_nodata=-999999.0, valid min 984.24 -> 984.24, valid max 1012.06 -> 1012.06, written=True
USGS_one_meter_x26y379_TX_Panhandle_B8_2017.tif: invalid=10072072, original_nodata=-999999.0, valid min 996.53 -> 996.53, valid max 1019.58 -> 1019.58, written=True
USGS_one_meter_x27y375_TX_Panhandle_B8_2017.tif: invalid=60579336, original_nodata=-999999.0, valid min 953.34 -> 953.34, valid max 973.76 -> 973.76, written=True
USGS_one_meter_x27y376_TX_Panhandle_B8_

In [8]:
for path in paths:
    with rasterio.open(path) as src:
        data = src.read(1, masked=True)
        valid = data.compressed()
        remaining_sentinels = np.count_nonzero(
            np.isin(data.data, tuple(INVALID_SENTINELS))
            & ~np.ma.getmaskarray(data)
        )
        print(
            f"{path.name}: metadata_nodata={src.nodata}, "
            f"valid min={valid.min():.6g}, valid max={valid.max():.6g}, "
            f"remaining sentinels={remaining_sentinels}, "
            f"masked pixels={int(np.ma.count_masked(data))}"
        )


USGS_one_meter_x26y375_TX_Panhandle_B8_2017.tif: metadata_nodata=-999999.0, valid min=966.95, valid max=991.86, remaining sentinels=0, masked pixels=61941281
USGS_one_meter_x26y376_TX_Panhandle_B8_2017.tif: metadata_nodata=-999999.0, valid min=968.2, valid max=998.66, remaining sentinels=0, masked pixels=0
USGS_one_meter_x26y377_TX_Panhandle_B8_2017.tif: metadata_nodata=-999999.0, valid min=969.63, valid max=1003.88, remaining sentinels=0, masked pixels=0
USGS_one_meter_x26y378_TX_Panhandle_B8_2017.tif: metadata_nodata=-999999.0, valid min=984.24, valid max=1012.06, remaining sentinels=0, masked pixels=0
USGS_one_meter_x26y379_TX_Panhandle_B8_2017.tif: metadata_nodata=-999999.0, valid min=996.53, valid max=1019.58, remaining sentinels=0, masked pixels=10072072
USGS_one_meter_x27y375_TX_Panhandle_B8_2017.tif: metadata_nodata=-999999.0, valid min=953.34, valid max=973.76, remaining sentinels=0, masked pixels=60579336
USGS_one_meter_x27y376_TX_Panhandle_B8_2017.tif: metadata_nodata=-99999